# 1.2 Catálogo Exaustivo de Fitness Landscapes — 115 Problemas de Benchmark

## Motivação e Contexto

A avaliação rigorosa de qualquer **Algoritmo Evolutivo Assistido por Surrogate (SAEA)** depende de problemas
de teste padronizados capazes de mimetizar as idiossincrasias encontradas em cenários reais de otimização
caixa-preta (*black-box optimization*). Este notebook implementa e analisa visualmente **todas as funções**
das 5 suítes de benchmark mais influentes na literatura de otimização contínua e multiobjetivo:

| Suíte | Ano | Autores | Problemas | Obj. | Total |
|-------|-----|---------|-----------|------|-------|
| **ZDT** | 2000 | Zitzler, Deb & Thiele | ZDT1–4, ZDT6 | 2 | 5 |
| **DTLZ** | 2005 | Deb, Thiele, Laumanns & Zitzler | DTLZ1–7 | 3 | 7 |
| **WFG** | 2006 | Huband, Hingston, Barone & While | WFG1–9 | 2 | 9 |
| **MMF** | 2019–2020 | Yue, Qu, Liang *et al.* (CEC) | MMF1–15 | 2 | 15 |
| **BBOB mono** | 2009– | Hansen, Finck, Ros, Auger (COCO) | F1–F24 | 1 | 24 |
| **BBOB-biobj** | 2016 | Tusar, Brockhoff *et al.* (COCO) | F1–F55 | 2 | 55 |
| | | | | **Total** | **115** |

### Por que estas suítes?

Cada suíte ataca uma faceta diferente do problema de otimização:

- **ZDT** — O *rito de passagem* clássico. Testa geometria da fronteira (convexa, côncava, desconectada)
  e multimodalidade separável. Arquitectura modular: $f_1(x_1)$, $g(x_{2:n})$, $h(f_1, g)$.

- **DTLZ** — Escalabilidade de objetivos ($M \geq 3$). Mapeia coordenadas esféricas no espaço de objetivos.
  Permite testar muitos-objetivos (*many-objective*) com geometrias controladas.

- **WFG** — O *pesadelo* dos surrogates. Pipeline de transformações sequenciais (shift → bias → reduction)
  que destroem a correlação entre variáveis de decisão e resposta. Epistasia extrema.

- **MMF** — Multimodalidade no espaço de *decisão*: múltiplos Conjuntos de Pareto (PS) mapeiam para a
  *mesma* Fronteira de Pareto (PF). Exige niching ativo no modelo de ML.

- **BBOB** — 24 funções mono-objetivo estratificadas em 5 classes topológicas. Translação do ótimo,
  rotações, e transformações não-lineares ($T_{osz}$, $T_{asy}$) tornam cada instância única.

- **BBOB-biobj** — 55 combinações bi-objetivo de 10 funções-base BBOB. Testa conflitos inter-objetivo
  onde um eixo é suave e o outro é fractal/rotacionado/caótico.

### Impacto nos Modelos de Machine Learning (Surrogates)

| Propriedade topológica | Efeito no XGBoost | Efeito no Kriging (GP) |
|------------------------|-------------------|----------------------|
| **Rotação** | Partições axis-parallel falham | Kernel RBF funciona, mas length-scale sofre |
| **Ill-conditioning** ($10^6$) | Feature importance colapsa | Convergência de hiperparâmetros falha |
| **Rugosidade fractal** | Splits infinitos necessários | Kernel Matérn 1/2 necessário |
| **Descontinuidade (PF)** | Adapta naturalmente | Violação de estacionaridade |
| **Epistasia (WFG)** | Profundidade de árvore explode | Matrizes de covariância cheias necessárias |
| **Multimodalidade (PS)** | Requer particionamento | Clustering ativo mandatório |

---

### Configurações Experimentais

Para todos os problemas, utilizamos:
- **Amostragem Sobol** (quasi-random, baixa discrepância) para problemas de alta dimensão
- **Meshgrid** para problemas com 2 variáveis de decisão (visualização completa)
- **Non-Dominated Sorting** (NDS) para extração da fronteira de Pareto empírica
- Dados salvos em `data/dataframes/{NOME}/` como Parquet

---

### Fronteiras de Pareto Verdadeiras (Analíticas)

Em todos os gráficos de fronteira de Pareto, uma **linha preta fina** representa a
**fronteira de Pareto analítica/teórica verdadeira** do problema, obtida por:

- **ZDT, DTLZ, WFG**: Equações analíticas via `pymoo.problem.pareto_front()`
- **MMF**: Derivação manual fixando $g(\mathbf{x}) = 0$ no Pareto Set teórico
- **BBOB mono**: $f^* = 0$ como referência (ótimo global por construção)
- **BBOB-biobj**: Fronteira de referência via amostra densa (500K pts) + NDS

Isto permite avaliar visualmente o quanto a fronteira empírica (amostrada) se **distancia**
da fronteira teórica verdadeira — especialmente relevante em problemas de alta dimensionalidade
onde a amostragem Sobol é esparsa.

In [ ]:
import pandas as pd
import numpy as np
import os, math, warnings, itertools
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors

from sklearn.decomposition import PCA
from scipy.stats import qmc, ortho_group
from pymoo.problems import get_problem
from pymoo.core.problem import Problem
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['lines.linewidth'] = 0.5
np.random.seed(42)

## Funções Utilitárias

Funções de geração de dados, busca de Pareto e visualização reutilizáveis em todo o notebook.

In [ ]:
# ======================================================================
# Data Generation
# ======================================================================
def eval_problem(problem, X):
    if X.ndim == 1: X = X.reshape(1, -1)
    out = {}; problem._evaluate(X, out)
    F = out["F"]
    return F.reshape(-1, 1) if F.ndim == 1 else F

def gen_grid_2d(problem, n_obs=500_000):
    n = int(np.sqrt(n_obs))
    g = [np.linspace(problem.xl[i], problem.xu[i], n) for i in range(2)]
    X0, X1 = np.meshgrid(*g); X = np.column_stack([X0.ravel(), X1.ravel()])
    F = eval_problem(problem, X)
    df = pd.DataFrame(X, columns=['x_1','x_2'])
    for j in range(F.shape[1]): df[f'f{j+1}'] = F[:, j]
    df['aux'] = 1; return df

def gen_sobol(problem, n_obs=100_000, alpha_trick=False):
    nv = problem.n_var
    sampler = qmc.Sobol(d=nv, scramble=True); Xr = sampler.random(n=n_obs)
    if alpha_trick:
        a = np.linspace(0, 1, n_obs); np.random.shuffle(a)
        for i in range(1, nv): Xr[:, i] *= a
    X = problem.xl + Xr * (problem.xu - problem.xl)
    F = eval_problem(problem, X)
    cols = [f'x_{i+1}' for i in range(nv)]
    df = pd.DataFrame(X, columns=cols)
    for j in range(F.shape[1]): df[f'f{j+1}'] = F[:, j]
    pca = PCA(n_components=1); df['_p'] = pca.fit_transform(X)
    df = df.sort_values('_p').reset_index(drop=True).drop(columns=['_p'])
    df['registro'] = df.index; df['aux'] = 1; return df

# ======================================================================
# Pareto / Best
# ======================================================================
def find_pareto(df, obj_cols, minimize=True):
    F = df[obj_cols].values
    if not minimize: F = -F
    nds = NonDominatedSorting(); fronts = nds.do(F); idx = fronts[0]
    return df.iloc[idx].copy().sort_values(obj_cols[0], ascending=minimize)

def find_best(df, col='f1', n=500):
    return df.nsmallest(n, col).copy()

def save_prob(name, df, dfp, base='data/dataframes'):
    d = os.path.join(base, name); os.makedirs(d, exist_ok=True)
    df.to_parquet(os.path.join(d, f'df_{name.lower()}.parquet'))
    dfp.to_parquet(os.path.join(d, f'df_pareto_{name.lower()}.parquet'))

# ======================================================================
# TRUE Pareto Front Retrieval
# ======================================================================

def get_true_pf_pymoo(prob, n_points=500):
    '''Get analytical PF from pymoo problem. Returns (n, n_obj) array or None.'''
    try:
        pf = prob.pareto_front(n_pareto_points=n_points)
        if pf is not None and len(pf) > 0:
            return pf
    except Exception:
        pass
    try:
        pf = prob.pareto_front()
        if pf is not None and len(pf) > 0:
            return pf
    except Exception:
        pass
    return None

def get_true_pf_mmf(name, n_points=500):
    '''Compute analytical PF for MMF problems by setting g=0 on the Pareto Set.'''
    f1 = np.linspace(1e-6, 1.0 - 1e-6, n_points)
    pf_formulas = {
        'MMF1':  lambda t: 1 - np.sqrt(t),
        'MMF2':  lambda t: 1 - np.sqrt(t),
        'MMF3':  lambda t: 1 - t,
        'MMF4':  lambda t: 1 - t**2,
        'MMF5':  lambda t: 1 - np.sqrt(t),
        'MMF6':  lambda t: 1 - np.sqrt(t),
        'MMF7':  lambda t: 1 - np.sqrt(t),
        'MMF9':  lambda t: 1 - np.sqrt(t),
        'MMF10': lambda t: 1 - np.sqrt(t),
        'MMF11': lambda t: 1 - np.sqrt(t),
        'MMF12': lambda t: 1 - np.sqrt(t),
        'MMF13': lambda t: 1 - np.sqrt(t),
        'MMF14': lambda t: 1 - np.sqrt(t),
        'MMF15': lambda t: 1 - t**2,
    }
    if name == 'MMF8':
        return np.array([[-1.0, -1.0]])
    func = pf_formulas.get(name)
    if func is None:
        return None
    return np.column_stack([f1, func(f1)])

def get_true_pf_bbob_biobj(cls1, cls2, n_var=5, seed1=0, seed2=1, n_obs=500_000):
    '''Approximate true PF for BBOB-biobj via dense sampling + NDS.'''
    from scipy.stats import qmc as _qmc
    sampler = _qmc.Sobol(d=n_var, scramble=True)
    Xr = sampler.random(n=n_obs)
    X = -5.0 + Xr * 10.0
    p1 = cls1(n_var=n_var, seed=seed1)
    p2 = cls2(n_var=n_var, seed=seed2)
    f1 = eval_problem(p1, X).ravel()
    f2 = eval_problem(p2, X).ravel()
    F = np.column_stack([f1, f2])
    nds = NonDominatedSorting()
    fronts = nds.do(F)
    pf = F[fronts[0]]
    pf = pf[pf[:, 0].argsort()]
    return pf

In [ ]:
# ======================================================================
# Visualization helpers — with TRUE PF overlay (thin black line)
# ======================================================================

def _front_ax(ax, df, dfp, title, true_pf=None, s_all=8, s_par=40, sample=50_000):
    '''Single Pareto front subplot with optional true PF overlay.'''
    n = min(sample, len(df)); samp = df.sample(n=n, random_state=42)
    ax.scatter(samp['f1'], samp['f2'], c='lightgray', s=s_all, alpha=0.2, rasterized=True)
    if true_pf is not None and len(true_pf) > 1:
        idx = true_pf[:, 0].argsort()
        ax.plot(true_pf[idx, 0], true_pf[idx, 1], 'k-', lw=1.5, alpha=0.9,
                label='PF verdadeira', zorder=4)
    elif true_pf is not None and len(true_pf) == 1:
        ax.plot(true_pf[0, 0], true_pf[0, 1], 'k*', ms=12, zorder=4, label='Otimo global')
    ax.scatter(dfp['f1'], dfp['f2'], c='red', s=s_par, alpha=0.8,
               edgecolors='darkred', linewidth=0.8, zorder=5)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('$f_1$', fontsize=8); ax.set_ylabel('$f_2$', fontsize=8)
    ax.grid(True, alpha=0.2)

def plot_front_grid(data, n_cols=5, suptitle='', figw=4, figh=3.5):
    '''Grid of 2-D Pareto fronts. data[name] = (df, dfp) or (df, dfp, true_pf).'''
    names = list(data.keys()); n = len(names)
    nr = math.ceil(n / n_cols)
    fig, axes = plt.subplots(nr, n_cols, figsize=(figw*n_cols, figh*nr))
    if nr == 1: axes = axes.reshape(1, -1)
    fig.suptitle(suptitle, fontsize=18, fontweight='bold', y=1.01)
    for i, name in enumerate(names):
        r, c = divmod(i, n_cols)
        entry = data[name]
        df, dfp = entry[0], entry[1]
        tpf = entry[2] if len(entry) > 2 else None
        _front_ax(axes[r, c], df, dfp, name, true_pf=tpf)
    for j in range(i+1, nr*n_cols):
        r, c = divmod(j, n_cols); axes[r, c].axis('off')
    plt.tight_layout(); plt.show(); return fig

def plot_front_3d_grid(data, n_cols=4, suptitle=''):
    '''Grid of 3-D Pareto fronts with true PF surface/points.'''
    names = list(data.keys()); n = len(names)
    nr = math.ceil(n / n_cols)
    fig = plt.figure(figsize=(5*n_cols, 5*nr))
    fig.suptitle(suptitle, fontsize=18, fontweight='bold', y=1.01)
    for i, name in enumerate(names):
        entry = data[name]
        df, dfp = entry[0], entry[1]
        tpf = entry[2] if len(entry) > 2 else None
        ax = fig.add_subplot(nr, n_cols, i+1, projection='3d')
        samp = df.sample(min(20_000, len(df)), random_state=42)
        ax.scatter(samp['f1'], samp['f2'], samp['f3'], c='lightgray', s=2, alpha=0.03)
        if tpf is not None and len(tpf) > 1:
            ax.scatter(tpf[:, 0], tpf[:, 1], tpf[:, 2], c='black', s=3, alpha=0.4,
                       label='PF verdadeira', zorder=4)
        ax.scatter(dfp['f1'], dfp['f2'], dfp['f3'], c='red', s=15, alpha=0.6, edgecolors='darkred')
        ax.set_title(name, fontsize=10, fontweight='bold')
        ax.set_xlabel('$f_1$',fontsize=7); ax.set_ylabel('$f_2$',fontsize=7); ax.set_zlabel('$f_3$',fontsize=7)
        ax.view_init(25, 135)
    plt.tight_layout(); plt.show(); return fig

def plot_mono_grid(data, n_cols=6, suptitle=''):
    names = list(data.keys()); n = len(names)
    nr = math.ceil(n / n_cols)
    fig, axes = plt.subplots(nr, n_cols, figsize=(3.5*n_cols, 3*nr))
    if nr == 1 and n_cols == 1: axes = np.array([[axes]])
    elif nr == 1: axes = axes.reshape(1, -1)
    elif n_cols == 1: axes = axes.reshape(-1, 1)
    fig.suptitle(suptitle, fontsize=16, fontweight='bold', y=1.01)
    for i, name in enumerate(names):
        r, c = divmod(i, n_cols); ax = axes[r, c]
        df, dbest = data[name][0], data[name][1]
        ax.hist(df['f1'], bins=60, color='steelblue', alpha=0.7, edgecolor='none')
        ax.axvline(dbest['f1'].max(), color='red', ls='--', lw=1.5, label='Top-n')
        ax.axvline(0, color='black', ls='-', lw=1.2, alpha=0.8, label='$f^*=0$')
        ax.set_title(name, fontsize=9, fontweight='bold')
        ax.set_xlabel('$f_1$', fontsize=7); ax.tick_params(labelsize=6)
    for j in range(i+1, nr*n_cols):
        r, c = divmod(j, n_cols); axes[r, c].axis('off')
    plt.tight_layout(); plt.show(); return fig

def plot_heatmap_grid(data, n_cols=5, suptitle='', obj='f1'):
    names = list(data.keys()); n = len(names)
    nr = math.ceil(n / n_cols)
    fig, axes = plt.subplots(nr, n_cols, figsize=(4*n_cols, 3.5*nr))
    if nr == 1: axes = axes.reshape(1, -1)
    fig.suptitle(suptitle, fontsize=16, fontweight='bold', y=1.01)
    for i, name in enumerate(names):
        r, c = divmod(i, n_cols); ax = axes[r, c]
        df, dfp = data[name][0], data[name][1]
        sc = ax.scatter(df['x_1'], df['x_2'], c=df[obj], cmap='plasma', s=1, alpha=0.5, rasterized=True)
        ax.scatter(dfp['x_1'], dfp['x_2'], c='white', edgecolor='black', s=8, lw=0.3, zorder=5)
        ax.set_title(name, fontsize=9, fontweight='bold')
        ax.set_xlabel('$x_1$', fontsize=7); ax.set_ylabel('$x_2$', fontsize=7)
        ax.tick_params(labelsize=6)
    for j in range(i+1, nr*n_cols):
        r, c = divmod(j, n_cols); axes[r, c].axis('off')
    plt.tight_layout(); plt.show(); return fig

def plot_dashboard(df, dfp, feats, title, n_obj=2, true_pf=None):
    '''4-panel dashboard with optional true PF overlay.'''
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.98)
    gs = GridSpec(2, 2, figure=fig, hspace=0.30, wspace=0.20)
    is2d = len(feats) == 2
    ax1 = fig.add_subplot(gs[0, 0])
    if is2d:
        sc = ax1.scatter(df[feats[0]], df[feats[1]], c=df['f1']+df.get('f2', 0),
                         cmap='viridis', s=3, alpha=0.4, rasterized=True)
        ax1.scatter(dfp[feats[0]], dfp[feats[1]], c='red', edgecolor='darkred', s=30, zorder=5, label='Pareto')
        ax1.set_xlabel(feats[0]); ax1.set_ylabel(feats[1]); ax1.legend()
    else:
        pca = PCA(n_components=2); Xp = pca.fit_transform(df[feats].values)
        ve = pca.explained_variance_ratio_*100
        sc = ax1.scatter(Xp[:,0], Xp[:,1], c=df['f1']+df.get('f2', 0),
                         cmap='viridis', s=5, alpha=0.3, rasterized=True)
        ax1.set_xlabel(f'PC1 ({ve[0]:.1f}%)'); ax1.set_ylabel(f'PC2 ({ve[1]:.1f}%)')
    ax1.set_title('Espaço de Decisão', fontsize=13, fontweight='bold')
    fig.colorbar(sc, ax=ax1)
    ax2 = fig.add_subplot(gs[0, 1])
    ns = min(50_000, len(df)); samp = df.sample(ns, random_state=42)
    if n_obj >= 2:
        ax2.scatter(samp['f1'], samp['f2'], c='lightgray', s=10, alpha=0.2, rasterized=True)
        if true_pf is not None and len(true_pf) > 1:
            idx = true_pf[:, 0].argsort()
            ax2.plot(true_pf[idx, 0], true_pf[idx, 1], 'k-', lw=2, alpha=0.9,
                     label='PF verdadeira', zorder=4)
        elif true_pf is not None and len(true_pf) == 1:
            ax2.plot(true_pf[0, 0], true_pf[0, 1], 'k*', ms=15, zorder=4, label='Otimo global')
        ax2.scatter(dfp['f1'], dfp['f2'], c='red', s=40, edgecolors='darkred', lw=1, alpha=0.8, zorder=5,
                    label=f'Empirico ({len(dfp):,})')
        ax2.set_xlabel('$f_1$'); ax2.set_ylabel('$f_2$')
        ax2.legend(fontsize=10)
    else:
        ax2.hist(df['f1'], bins=80, color='steelblue', alpha=0.7, edgecolor='none')
        ax2.axvline(dfp['f1'].max(), color='red', ls='--', lw=2, label=f'Top-{len(dfp)}')
        ax2.axvline(0, color='black', ls='-', lw=1.5, label='$f^*=0$')
        ax2.set_xlabel('$f_1$'); ax2.legend(fontsize=10)
    ax2.set_title('Espaço de Objetivos', fontsize=13, fontweight='bold'); ax2.grid(True, alpha=0.3)
    nplot = min(100, len(dfp))
    pp = dfp.sample(nplot, random_state=42).sort_values('f1') if len(dfp) > nplot else dfp.sort_values('f1')
    xlabels = feats if len(feats) <= 12 else [f'$x_{{{i+1}}}$' for i in range(len(feats))]
    ax3 = fig.add_subplot(gs[1, 0])
    norm = Normalize(vmin=pp['f1'].min(), vmax=pp['f1'].max()); cm = plt.get_cmap('cool')
    for _, row in pp.iterrows():
        ax3.plot(range(len(feats)), row[feats], color=cm(norm(row['f1'])), alpha=0.7, lw=1.2)
    ax3.set_xticks(range(len(feats))); ax3.set_xticklabels(xlabels, rotation=45 if len(feats)>5 else 0, fontsize=7)
    ax3.set_title('Coordenadas Paralelas', fontsize=13, fontweight='bold')
    sm = ScalarMappable(cmap=cm, norm=norm); sm.set_array([]); fig.colorbar(sm, ax=ax3, label='$f_1$')
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.boxplot(pp[feats].values, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7), medianprops=dict(color='red', lw=2))
    ax4.set_xticklabels(xlabels, rotation=45 if len(feats)>5 else 0, fontsize=7)
    ax4.set_title('Boxplots (Pareto/Best)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show(); return fig

---
# 1. Suíte ZDT — Zitzler, Deb & Thiele (2000)

## Origem Histórica

Proposta por **Eckart Zitzler**, **Kalyanmoy Deb** e **Lothar Thiele** no artigo seminal
*"Comparison of Multiobjective Evolutionary Algorithms: Empirical Results"* (Evolutionary Computation, 2000),
a suíte ZDT tornou-se o **rito de passagem obrigatório** para qualquer novo Algoritmo Evolutivo Multiobjetivo
(MOEA), incluindo o NSGA-II e o SPEA2.

## Arquitectura Modular

Todos os problemas ZDT compartilham a mesma estrutura algébrica de três funções:

$$\text{Minimizar} \quad F(\mathbf{x}) = \big(f_1(x_1),\; f_2(\mathbf{x})\big)$$

onde:

- $f_1(x_1)$: controla a dispersão ao longo do primeiro eixo objetivo (depende apenas de $x_1$)
- $g(x_2, \ldots, x_n)$: mede a **dificuldade de convergência** (depende das variáveis restantes)
- $h(f_1, g)$: define a **geometria** da fronteira de Pareto (convexa, côncava, desconectada)
- $f_2 = g \cdot h$

A separabilidade intencional ($x_1$ isolado de $x_{2:n}$) torna a suíte um **grupo de controle** ideal
para testar se novos mecanismos de busca não degradam a convergência basal.

> **Nota:** ZDT5 (representação binária) foi excluído deste catálogo por não se aplicar a otimização contínua.

## Descrição Individual dos Problemas

| Prob. | $D$ | Geometria PF | Modalidade | Descrição |
|-------|-----|-------------|------------|-----------|
| **ZDT1** | 30 | Convexa | Unimodal | **Baseline universal.** $g = 1 + 9\bar{x}$, $h = 1 - \sqrt{f_1/g}$. Superfície suave, separável. Qualquer surrogate competente deve resolvê-lo. |
| **ZDT2** | 30 | Côncava | Unimodal | $h = 1 - (f_1/g)^2$. Formato não-convexo testa se o algoritmo converge para zonas de trade-off sem colapsar no centro da fronteira. |
| **ZDT3** | 30 | **Desconectada** (5 seg.) | Multimodal em $h$ | Perturbação senoidal $h = 1 - \sqrt{f_1/g} - (f_1/g)\sin(10\pi f_1)$. A fronteira se divide em **5 segmentos disjuntos**. Modelos GP com kernels estacionários falham ao tentar interpolar os "buracos". |
| **ZDT4** | 10 | Convexa | **$21^9$ ótimos locais** | $g$ incorpora a equação de **Rastrigin**: $g = 1 + 10(n-1) + \sum(x_i^2 - 10\cos 4\pi x_i)$. Armadilha massiva de ótimos locais paralelos à fronteira verdadeira. |
| **ZDT6** | 10 | Côncava | Não-uniforme | $f_1 = 1 - \exp(-4x_1)\sin^6(6\pi x_1)$, $g = 1 + 9(\bar{x})^{0.25}$. **Viés severo de distribuição**: a densidade de soluções cai drasticamente perto da fronteira ótima, enviesando regressões. |

In [ ]:
print('='*60); print('SUITE ZDT (5 problemas)'); print('='*60)
zdt_cfg = {
    'ZDT1': dict(n_var=30, at=True),
    'ZDT2': dict(n_var=30, at=True),
    'ZDT3': dict(n_var=30, at=True),
    'ZDT4': dict(n_var=10, at=False),
    'ZDT6': dict(n_var=10, at=True),
}
zdt = {}
for name, c in zdt_cfg.items():
    prob = get_problem(name.lower(), n_var=c['n_var'])
    df = gen_sobol(prob, n_obs=100_000, alpha_trick=c['at'])
    dfp = find_pareto(df, ['f1','f2'])
    tpf = get_true_pf_pymoo(prob)
    save_prob(name, df, dfp)
    zdt[name] = (df, dfp, tpf)
    pf_status = f'{len(tpf)} pts' if tpf is not None else 'N/A'
    print(f"  {name}: {len(df):,} pts, Pareto emp. {len(dfp):,}, PF verdadeira {pf_status}")

In [ ]:
plot_front_grid(zdt, n_cols=3, suptitle='Suíte ZDT — Fronteiras de Pareto (linha preta = PF verdadeira)')

In [ ]:
for name in ['ZDT1', 'ZDT3']:
    df, dfp, tpf = zdt[name]
    feats = [f'x_{i+1}' for i in range(df.shape[1] - 4)]
    plot_dashboard(df, dfp, feats, f'{name} — Dashboard ({len(feats)}D)', true_pf=tpf)

---
# 2. Suíte DTLZ — Deb, Thiele, Laumanns & Zitzler (2005)

## Origem Histórica

Publicada por **Kalyanmoy Deb**, **Lothar Thiele**, **Marco Laumanns** e **Eckart Zitzler** em
*"Scalable Test Problems for Evolutionary Multiobjective Optimization"* (2005), a suíte DTLZ foi criada
para superar a principal limitação da ZDT: a **incapacidade de escalar o número de objetivos**.

## Filosofia de Construção

A suíte DTLZ utiliza **coordenadas hiperesféricas** para mapear as variáveis de decisão ao espaço de
objetivos. As variáveis se dividem em dois grupos:

- **Variáveis de posição** ($x_1, \ldots, x_{M-1}$): controlam os ângulos de varredura na hiperesfera,
  determinando a posição na fronteira.
- **Variáveis de distância** ($x_M, \ldots, x_n$): controlam o raio $g(\mathbf{x})$, penalizando
  soluções longe da fronteira ótima (onde $g = 0$).

A fórmula geral é:
$$f_m(\mathbf{x}) = (1 + g(\mathbf{x})) \prod_{i=1}^{M-m} \cos(\theta_i) \cdot \begin{cases} 1 & m=1 \\ \sin(\theta_{M-m+1}) & m>1 \end{cases}$$

onde $\theta_i = \frac{\pi}{2} x_i$ para DTLZ2/3/4.

## Descrição Individual (todos com $M=3$ objetivos neste catálogo)

| Prob. | $D$ | Geometria PF | Modalidade | Descrição |
|-------|-----|-------------|------------|-----------|
| **DTLZ1** | 7 | **Hiperplano linear** | Massiv. multimodal | $g$ usa Rastrigin: $(M-1)\cdot 100 + \sum[\ldots]$. A PF é $\sum f_m = 0.5$. Milhares de ótimos locais criam falsas fronteiras paralelas. |
| **DTLZ2** | 12 | **Esférica côncava** (1/8) | Unimodal | **Baseline tri-objetivo.** PF é a casca unitária no quadrante positivo: $\sum f_m^2 = 1$. Convergência suave, perfeitamente modelável por redes neurais contínuas. |
| **DTLZ3** | 12 | Esférica côncava | **Massiv. multimodal** | Mesma geometria do DTLZ2, mas $g$ herdado do DTLZ1. Testa a distinção entre predição global e regressão de ruído paramétrico. |
| **DTLZ4** | 12 | Esférica (bias) | Unimodal + bias | Idêntico ao DTLZ2, porém $\theta_i = \frac{\pi}{2}x_i^\alpha$ com $\alpha=100$. O **viés severo** concentra >99% das soluções num canto da fronteira. Regressões uniformes estimam áreas vazias como ricas. |
| **DTLZ5** | 12 | **Degenerada** (curva) | Unimodal | A PF colapsa para uma **curva 1D** independente do número de objetivos. $\theta_i = \frac{\pi}{4g+2}(1+2g\cdot x_i)$. Testa se o surrogate aprende o manifold de dimensionalidade reduzida. |
| **DTLZ6** | 12 | Degenerada (curva) | Unimodal + bias | Como DTLZ5, mas $g = \sum x_i^{0.1}$ cria gradientes difíceis de inferir em aproximações quadráticas independentes. |
| **DTLZ7** | 22 | **Desconectada** (4 regiões) | Multimodal | $f_M = (1+g)\big[M - \sum \frac{f_m}{1+g}(1+\sin 3\pi f_m)\big]$. A fronteira se fragmenta em **$2^{M-1}$ regiões** desconexas. Surrogates globais tentarão interpolar zonas mortas impossíveis. |

In [ ]:
print('='*60); print('SUITE DTLZ (7 problemas, 3 obj)'); print('='*60)
dtlz_cfg = {
    'DTLZ1': dict(n_var=7, n_obj=3),
    'DTLZ2': dict(n_var=12, n_obj=3),
    'DTLZ3': dict(n_var=12, n_obj=3),
    'DTLZ4': dict(n_var=12, n_obj=3),
    'DTLZ5': dict(n_var=12, n_obj=3),
    'DTLZ6': dict(n_var=12, n_obj=3),
    'DTLZ7': dict(n_var=22, n_obj=3),
}
dtlz = {}
for name, c in dtlz_cfg.items():
    prob = get_problem(name.lower(), n_var=c['n_var'], n_obj=c['n_obj'])
    df = gen_sobol(prob, n_obs=100_000)
    dfp = find_pareto(df, ['f1','f2','f3'])
    tpf = get_true_pf_pymoo(prob)
    save_prob(name, df, dfp)
    dtlz[name] = (df, dfp, tpf)
    pf_status = f'{len(tpf)} pts' if tpf is not None else 'N/A'
    print(f"  {name}: {len(df):,} pts, Pareto emp. {len(dfp):,}, PF verdadeira {pf_status}")

In [ ]:
plot_front_3d_grid(dtlz, n_cols=4,
    suptitle='Suíte DTLZ — Fronteiras 3-D (preto = PF verdadeira, vermelho = empírica)')

In [ ]:
for name in ['DTLZ2', 'DTLZ7']:
    df, dfp, tpf = dtlz[name]
    feats = [f'x_{i+1}' for i in range(df.shape[1] - 5)]
    plot_dashboard(df, dfp, feats, f'{name} — Dashboard', n_obj=3, true_pf=tpf)

---
# 3. Suíte WFG — Walking Fish Group (Huband *et al.*, 2006)

## Origem Histórica

Desenvolvida por **Simon Huband**, **Philip Hingston**, **Luigi Barone** e **Lyndon While** da
Universidade da Austrália Ocidental (*Walking Fish Group*), publicada em
*"A Review of Multiobjective Test Problems and a Scalable Test Problem Toolkit"* (2006).

A WFG foi criada para **corrigir limitações remanescentes** das suítes ZDT e DTLZ, introduzindo
um paradigma revolucionário baseado em **transformações sequenciais**.

## Filosofia de Design: Pipeline de Transformações

Em vez de as variáveis de decisão mapearem diretamente ao espaço objetivo, a WFG utiliza um pipeline:

$$\mathbf{x} \xrightarrow{T_1} \mathbf{y}^{(1)} \xrightarrow{T_2} \mathbf{y}^{(2)} \xrightarrow{\ldots} \mathbf{y}^{(p)} \xrightarrow{\text{shape}} \mathbf{f}$$

As transformações incluem:

- **Shift** ($b\_flat$, $s\_linear$): Translação dos ótimos para posições não-triviais
- **Bias** ($b\_poly$, $b\_param$): Distorções polinomiais que comprimem 90% do espaço para 10% dos objetivos
- **Reduction** ($r\_sum$, $r\_nonsep$): Redução de dimensionalidade com **epistasia** não-separável
- **Shape** (convex, concave, linear, disconnected, mixed): Geometria final da PF

> **Para surrogates:** A WFG é o *pesadelo último*. A cascata de transformações destrói a correlação
> entre variáveis de decisão e resposta. XGBoost precisa de profundidade de árvore absurda;
> GPs precisam de matrizes de covariância totalmente parametrizáveis.

## Descrição Individual

| Prob. | Geom. PF | Modal. | Separ. | Descrição |
|-------|----------|--------|--------|-----------|
| **WFG1** | Convexa/mista | Unimodal | Sim | Bias polinomial forte + *flat regions*. Cria espaços rasos onde o gradiente zera, iludindo surrogates baseados em ganho de informação. |
| **WFG2** | Convexa descon. | Multimodal | **Não** | Redução não-separável cria dependências cruzadas. PF com furos descontínuos. XGBoost requer subdivisão imensa; GPs com RBF cheias lidam melhor. |
| **WFG3** | Linear degen. | Unimodal | Não | Anomalia proposital: PF colapsa para uma reta 1D independente de $M$. Simula perda de posto em problemas de engenharia real. |
| **WFG4** | Côncava | Mass. multi. | Sim | Colinas multimodais com raios de atração largos (*hill sizes*). Bacias gravitacionais de erro prendem otimizadores assistidos por surrogate. |
| **WFG5** | Côncava | **Deceptiva** | Sim | Vales atratores profundos geram sinalizações falso-negativas em gradientes. Surrogates gulosos colapsam nesta topologia. |
| **WFG6** | Côncava | Unimodal | **Não** | Epistasia algébrica extrema via $r\_nonsep$. Variáveis não suportam fixação linear separada. Demanda regressão matricial completa. |
| **WFG7** | Côncava | Unimodal | Sim | Correlação direcional: eixos de "posição" dependem dos parâmetros de "distância". Modelos de inferência causal obtêm vantagem. |
| **WFG8** | Côncava | Unimodal | **Não** | Clímax de inter-dependência combinatória. Bias de distância escalar acopla diferentes vetores, inviabilizando GP baseados apenas em L2 homogêneo. |
| **WFG9** | Côncava | **Deceptiva + multi.** | **Não** | A *"Tempestade Perfeita"*: deceptividade global + epistasia não-separável extrema + multimodalidade oscilatória. Obriga LHS rigoroso e surrogates profundos. |

In [ ]:
print('='*60); print('SUITE WFG (9 problemas)'); print('='*60)
wfg = {}
for i in range(1, 10):
    name = f'WFG{i}'
    prob = get_problem(name.lower(), n_var=10, n_obj=2)
    df = gen_sobol(prob, n_obs=100_000)
    dfp = find_pareto(df, ['f1','f2'])
    tpf = get_true_pf_pymoo(prob)
    save_prob(name, df, dfp)
    wfg[name] = (df, dfp, tpf)
    pf_status = f'{len(tpf)} pts' if tpf is not None else 'N/A'
    print(f"  {name}: {len(df):,} pts, Pareto emp. {len(dfp):,}, PF verdadeira {pf_status}")

In [ ]:
plot_front_grid(wfg, n_cols=3, suptitle='Suíte WFG — Fronteiras de Pareto (linha preta = PF verdadeira)')

In [ ]:
for name in ['WFG1', 'WFG9']:
    df, dfp, tpf = wfg[name]
    feats = [f'x_{i+1}' for i in range(10)]
    plot_dashboard(df, dfp, feats, f'{name} — Dashboard (10D)', true_pf=tpf)

---
# 4. Suíte MMF — Multi-Modal Multi-Objective Functions (CEC 2019/2020)

## Origem Histórica

A suíte MMF foi proposta no contexto das competições do **IEEE Congress on Evolutionary Computation (CEC)**:

- **CEC 2019**: *"Problem Definitions for the CEC 2019 Special Session on Multimodal Multi-Objective Optimization"*
  por **Caitong Yue**, **Boyang Qu**, **Jing Liang** *et al.*. Definiu MMF1–MMF8.
- **CEC 2020**: Extensão com MMF9–MMF15 para cenários escaláveis e 3D.

## Conceito Fundamental: Multimodalidade no Espaço de Decisão

A suíte MMF introduz um paradigma diferente dos benchmarks tradicionais. O desafio **não está** na
geometria da fronteira de Pareto (que pode ser simples), mas sim na existência de **múltiplos Conjuntos
de Pareto (PS)** no espaço de decisão que mapeiam para a **mesma** fronteira no espaço de objetivos:

$$\text{PS}_1, \text{PS}_2, \ldots, \text{PS}_k \;\;\xrightarrow{\;F(\mathbf{x})\;}\;\; \text{PF (única)}$$

> **Para surrogates:** O modelo de ML deve reconhecer que para um mesmo valor $(f_1, f_2)$, existem
> **múltiplos vetores $\mathbf{x}$** válidos. Redes neurais genéricas e modelos de regressão simples
> colapsam nesta ambiguidade. São necessárias técnicas de **niching ativo** e **particionamento Voronoi**
> no espaço de decisão.

## Descrição Individual

| Prob. | $D$ | PS | Descrição |
|-------|-----|----|-----------|
| **MMF1** | 2 | 2 PS espelhados | $f_1 = \|x_1-2\|$, PS definido por $x_2 = \sin(6\pi\|x_1-2\|+\pi)$. Duas regiões espelhadas em $x_1=2$. Baseline para provar visualmente o sucesso do niching. |
| **MMF2** | 2 | 2 PS piecewise | PS definido por funções cosseno por partes. Quebras abruptas de comportamento destroem o *smoothness prior* dos GP. |
| **MMF3** | 2 | 2 PS côncavos | Variante côncava do MMF1. Funis falsos locais repulsam otimizadores para frentes sub-ótimas. |
| **MMF4** | 2 | **4 PS** simétricos | $f_1 = \|x_1\|$, PS via $\sin(\pi(x_2 - \sin(\pi\|x_1\|)))$. Quatro regiões ótimas formando dois "M"s. Cenário supremo de diversidade inter-nicho. |
| **MMF5** | 2 | 2 PS + local | Usa $\min(f_1, 4(x_1-2)^2)$ para criar frente local competindo com a global. |
| **MMF6** | 2 | PS lineares | PS são retas ($x_2 = 0.5x_1 - 1$). Favorece PCA/PLS como preditores mas complica a convexidade. |
| **MMF7** | 2 | PS harmônicos | Harmônicos trigonométricos complexos dentro de $g$. Kernels periódicos de GP colapsam ao assumir previsibilidade cíclica. |
| **MMF8** | 2 | PS via $\sin$ | $f_1 = \sin(x_1)$, $f_2 = \sin(x_2)$ em $[-\pi, \pi]^2$. Periodicidade infinita cria platôs multimodais repetitivos onde o gradiente estagna. |
| **MMF9** | 2 | Repulsão concêntrica | Bacias em falsos PS locais obscurecem o acesso ao PS global. Impede equilíbrio exploração/explotação no Kriging. |
| **MMF10** | 2 | **1 PS** linear | Baseline unitário: $x_2 = \sin(\pi x_1)$. Calibração métrica simplificada por OLS. |
| **MMF11** | 2 | $l$ PS escaláveis | O número de PS é controlável via parâmetro $l$. Permite avaliar GMM incremental. |
| **MMF12** | 2 | PS descontínuos | Segmentação discreta controlada por $k$. Random Forests adaptam nativamente; Kriging contínuo sucumbe. |
| **MMF13** | **3** | PS não-linear 3D | Primeira extensão a 3 variáveis. Epistasia restritiva conjunta arruína regressões independentes por eixo. |
| **MMF14** | **4** | Escalável $(M,D)$ | Expansão hiper-geométrica escalável. Rampa restritiva final para Auto-Encoders dinâmicos. |
| **MMF15** | 4 | Escalável + atratores | Como MMF14, mas com falsos atratores locais explícitos. Engana métricas estocásticas de Otimização Bayesiana. |

In [ ]:
# ==============================================================================
# MMF Problem Definitions (CEC 2019 / 2020)
# ==============================================================================

class MMF1(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([1.,-1.]), xu=np.array([3.,1.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = np.abs(x1-2)
        s = np.sin(6*np.pi*np.abs(x1-2)+np.pi)
        out["F"] = np.column_stack([f1, 1-np.sqrt(f1)+2*(x2-s)**2])

class MMF2(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([-1.,0.]), xu=np.array([1.,2.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = np.abs(x1)
        g_star = np.where(x1>=0, np.cos(0.5*np.pi*x1), np.cos(0.5*np.pi*(-x1))+1)
        out["F"] = np.column_stack([f1, 1-np.sqrt(f1)+2*(x2-g_star)**2])

class MMF3(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([-1.,0.]), xu=np.array([1.,2.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = np.abs(x1)
        g_star = np.where(x1>=0, np.cos(0.5*np.pi*x1), -np.cos(0.5*np.pi*x1)+1)
        out["F"] = np.column_stack([f1, 1-f1+2*(x2-g_star)**2])

class MMF4(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([-1.,0.]), xu=np.array([1.,2.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = np.abs(x1)
        out["F"] = np.column_stack([f1, 1-x1**2+2*np.sin(np.pi*(x2-np.sin(np.pi*np.abs(x1))))**2])

class MMF5(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([1.,-1.]), xu=np.array([3.,1.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = np.abs(x1-2)
        s = np.sin(6*np.pi*np.abs(x1-2)+np.pi)
        g = 2*(x2-s)**2; f2 = np.minimum(f1, 4*(x1-2)**2) + g
        out["F"] = np.column_stack([f1, f2])

class MMF6(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([1.,-1.]), xu=np.array([3.,1.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = np.abs(x1-2)
        g_star = 0.5*x1 - 1
        out["F"] = np.column_stack([f1, 1-np.sqrt(f1)+2*(x2-g_star)**2])

class MMF7(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([1.,-1.]), xu=np.array([3.,1.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = np.abs(x1-2)
        s = np.sin(2*np.pi*np.abs(x1-2)+np.pi/2)
        pen = 1 + 0.5*np.abs(np.sin(4*np.pi*np.abs(x1-2)))
        out["F"] = np.column_stack([f1, 1-np.sqrt(f1)+2*pen*(x2-s)**2])

class MMF8(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2,
                         xl=np.array([-np.pi,-np.pi]), xu=np.array([np.pi,np.pi]))
    def _evaluate(self, X, out, *args, **kwargs):
        out["F"] = np.column_stack([np.sin(X[:,0]), np.sin(X[:,1])])

class MMF9(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2,
                         xl=np.array([-1.,-1.]), xu=np.array([1.,1.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = x1**2
        g = 1 + 10*(x2 - np.sin(np.pi*x1))**2 + np.abs(np.cos(4*np.pi*x1))
        out["F"] = np.column_stack([f1, g*(1-np.sqrt(f1/g))])

class MMF10(Problem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, xl=np.array([0.,-1.]), xu=np.array([1.,1.]))
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = x1
        out["F"] = np.column_stack([f1, 1-np.sqrt(f1)+2*(x2-np.sin(np.pi*x1))**2])

class MMF11(Problem):
    def __init__(self, l=3):
        super().__init__(n_var=2, n_obj=2, xl=np.array([0.,-1.]), xu=np.array([1.,1.]))
        self.l = l
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = x1
        out["F"] = np.column_stack([f1, 1-np.sqrt(f1)+2*(x2-np.sin(self.l*np.pi*x1))**2])

class MMF12(Problem):
    def __init__(self, l=3, k=3):
        super().__init__(n_var=2, n_obj=2, xl=np.array([0.,-1.]), xu=np.array([1.,1.]))
        self.l = l; self.k = k
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2 = X[:,0], X[:,1]; f1 = x1
        g_star = np.sin(self.l*np.pi*x1)
        g_star = np.where(np.mod(np.floor(self.k*x1), 2)==0, g_star, -g_star)
        out["F"] = np.column_stack([f1, 1-np.sqrt(f1)+2*(x2-g_star)**2])

class MMF13(Problem):
    def __init__(self, l=3):
        super().__init__(n_var=3, n_obj=2,
                         xl=np.array([0.,-1.,-1.]), xu=np.array([1.,1.,1.]))
        self.l = l
    def _evaluate(self, X, out, *args, **kwargs):
        x1, x2, x3 = X[:,0], X[:,1], X[:,2]; f1 = x1
        g = 1 + 2*(x2-np.sin(self.l*np.pi*x1))**2 + 2*(x3-np.sin(self.l*np.pi*x1))**2
        out["F"] = np.column_stack([f1, g*(1-np.sqrt(f1/g))])

class MMF14(Problem):
    def __init__(self, n_var=4):
        super().__init__(n_var=n_var, n_obj=2,
                         xl=np.zeros(n_var), xu=np.ones(n_var))
    def _evaluate(self, X, out, *args, **kwargs):
        f1 = X[:,0]
        gsum = np.sum(2*(X[:,1:]-np.sin(np.pi*X[:,0:1]))**2, axis=1)
        g = 1 + gsum
        out["F"] = np.column_stack([f1, g*(1-np.sqrt(f1/g))])

class MMF15(Problem):
    def __init__(self, n_var=4):
        super().__init__(n_var=n_var, n_obj=2,
                         xl=np.zeros(n_var), xu=np.ones(n_var))
    def _evaluate(self, X, out, *args, **kwargs):
        f1 = X[:,0]
        gsum = np.sum(2*np.sin(np.pi*X[:,1:])**2 + 2*(X[:,1:]-np.sin(np.pi*X[:,0:1]))**2, axis=1)
        g = 1 + gsum
        out["F"] = np.column_stack([f1, g*(1-(f1/g)**2)])

In [ ]:
print('='*60); print('SUITE MMF (15 problemas)'); print('='*60)
mmf_classes = {
    'MMF1': MMF1(), 'MMF2': MMF2(), 'MMF3': MMF3(), 'MMF4': MMF4(),
    'MMF5': MMF5(), 'MMF6': MMF6(), 'MMF7': MMF7(), 'MMF8': MMF8(),
    'MMF9': MMF9(), 'MMF10': MMF10(), 'MMF11': MMF11(), 'MMF12': MMF12(),
    'MMF13': MMF13(), 'MMF14': MMF14(), 'MMF15': MMF15(),
}
mmf = {}
for name, prob in mmf_classes.items():
    if prob.n_var == 2:
        df = gen_grid_2d(prob, n_obs=250_000)
    else:
        df = gen_sobol(prob, n_obs=100_000)
    dfp = find_pareto(df, ['f1','f2'])
    tpf = get_true_pf_mmf(name)
    save_prob(name, df, dfp)
    mmf[name] = (df, dfp, tpf)
    pf_desc = f'{len(tpf)} pts' if tpf is not None else 'N/A'
    print(f"  {name}: {len(df):,} pts, Pareto emp. {len(dfp):,}, PF verdadeira {pf_desc}")

### Visualização: Heatmaps no Espaço de Decisão (problemas 2D)

Para os problemas MMF com apenas 2 variáveis de decisão, podemos visualizar diretamente a
topologia do espaço de decisão através de heatmaps coloridos pelo valor de cada objetivo.
Os pontos brancos indicam o **Pareto Set** — as regiões ótimas no espaço de decisão.

In [ ]:
mmf_2d = {k: v for k, v in mmf.items() if v[0].shape[1] >= 4 and 'x_2' in v[0].columns and 'x_3' not in v[0].columns}
if mmf_2d:
    plot_heatmap_grid(mmf_2d, n_cols=5, suptitle='MMF (2D) — Heatmap $f_1$ no Espaço de Decisão', obj='f1')
    plot_heatmap_grid(mmf_2d, n_cols=5, suptitle='MMF (2D) — Heatmap $f_2$ no Espaço de Decisão', obj='f2')

### Visualização: Fronteiras de Pareto no Espaço de Objetivos

Cada subplot mostra a nuvem de pontos amostrados (cinza) e a fronteira de Pareto empírica (vermelho).
Note como problemas como MMF8 apresentam fronteiras compactas (ponto único ou região pequena),
enquanto o verdadeiro desafio está na multiplicidade de **conjuntos de Pareto** no espaço de decisão.

In [ ]:
plot_front_grid(mmf, n_cols=5, suptitle='Suíte MMF — Fronteiras de Pareto (linha preta = PF verdadeira)')

In [ ]:
for name in ['MMF1', 'MMF4', 'MMF8', 'MMF13']:
    df, dfp, tpf = mmf[name]
    feats = [c for c in df.columns if c.startswith('x_')]
    plot_dashboard(df, dfp, feats, f'{name} — Dashboard ({len(feats)}D)', true_pf=tpf)

---
# 5. Suíte BBOB Clássica — Black-Box Optimization Benchmarking (COCO, 2009–)

## Origem Histórica

A suíte BBOB é a espinha dorsal da plataforma **COCO** (*Comparing Continuous Optimizers*), mantida
por **Nikolaus Hansen**, **Steffen Finck**, **Raymond Ros** e **Anne Auger** desde 2009. O framework é
utilizado anualmente em workshops de benchmarking no **GECCO** (Genetic and Evolutionary Computation Conference).

## Filosofia de Instâncias

Cada função BBOB gera **instâncias paramétricas infinitas** através de:
- **Translação do ótimo** ($\mathbf{x}^* \in [-4, 4]^D$): impede overfitting na origem
- **Translação do valor ótimo** ($f^*$): impede exploração de valores canônicos
- **Matrizes de rotação** ($\mathbf{R}, \mathbf{Q}$): criam epistasia por acoplamento dimensional

## Transformações Estruturais

- $T_{osz}(\mathbf{z})$: **Oscilação** não-linear — destrói regularidades e simetrias
- $T_{asy}(\mathbf{z}, \beta)$: **Assimetria** — penaliza variações positivas/negativas desigualmente
- $\Lambda^{\alpha}$: **Condicionamento** — cria escalas de sensibilidade exponencialmente diferentes entre dimensões

## As 5 Classes de Dificuldade (24 Funções, 5D)

### Classe 1: Separáveis (F1–F5)
| F | Nome | Descrição |
|---|------|-----------|
| **F1** | Sphere | $f = \sum z_i^2$. Perfeitamente isotrópica. Baseline absoluto — todo ML resolve. |
| **F2** | Ellipsoid Sep. | $f = \sum 10^{6i/(D-1)} z_i^2$. Condicionamento $10^6$ mas separável. Árvores resistem; gradientes explodem. |
| **F3** | Rastrigin Sep. | $10D + \sum(z_i^2 - 10\cos 2\pi z_i)$. Macro-funil retalhado por infinitas cavidades senoidais. |
| **F4** | Bueche-Rastrigin | Como F3 com assimetria (*skew*) severa nos eixos pares. Instabilidade em GP simétricos. |
| **F5** | Linear Slope | Ótimo fora dos limites do domínio. Provoca extrapolação cega em árvores regressivas. |

### Classe 2: Unimodal Moderada (F6–F9)
| F | Nome | Descrição |
|---|------|-----------|
| **F6** | Attractive Sector | Apenas ~1% do espaço contém convergência útil. XGBoost constrói partições estéreis no restante. |
| **F7** | Step-Ellipsoid | Truncamento $\lfloor\cdot\rfloor$ cria platôs discretos. Destrói gradientes contínuos necessários para GP/Newton. |
| **F8** | Rosenbrock | O clássico *banana valley*: vale estreito, longo, arqueado. Correlação cruzada extrema. |
| **F9** | Rosenbrock Rot. | F8 com rotação ortogonal. Força XGBoost a simular diagonalidade complexa — tentativa letal. |

### Classe 3: Unimodal Ill-Conditioned (F10–F14)
| F | Nome | Descrição |
|---|------|-----------|
| **F10** | Ellipsoid Rotated | F2 + rotação. Condicionamento $10^6$ + epistasia rotacional. Aniquila preditores axis-parallel. |
| **F11** | Discus | $10^6 z_1^2 + \sum z_i^2$. Uma dimensão domina por fator de 1 milhão. Feature importance colapsa. |
| **F12** | Bent Cigar | $z_1^2 + 10^6\sum z_i^2$. Inverso do Discus. Curvatura microscópica ao longo do eixo dominante. |
| **F13** | Sharp Ridge | $z_1^2 + 100\sqrt{\sum z_i^2}$. Cume descontínuo — gradiente tangencial inexistente. *Chattering* na crista. |
| **F14** | Different Powers | $\sum\|z_i\|^{2+4i/(D-1)}$. Expoentes heterogêneos destroem a premissa de distância L2 estacionária. |

### Classe 4: Multimodal com Estrutura Global (F15–F19)
| F | Nome | Descrição |
|---|------|-----------|
| **F15** | Rastrigin Rotated | F3 + rotação. Caos oscilatório no plano hiperesférico de simetria bloqueada. |
| **F16** | Weierstrass | **Fractal**: contínua em toda parte, diferenciável em nenhuma. Priors Gaussianos suaves colapsam. |
| **F17** | Schaffer F7 | Harmônicos de alta frequência cruzados. Cristas prendem PSO; surrogates precisam de kernels espectrais. |
| **F18** | Schaffer F7 Cond. | F17 com condicionamento $10^3$. Agrava a perda de tração em fases convergentes do GP. |
| **F19** | Griewank-Rosenbrock | Macro-vale Rosenbrock infestado de micro-picos Griewank. Armadilhas locais invisíveis a L2 regressivos. |

### Classe 5: Multimodal Fracamente Estruturada (F20–F24)
| F | Nome | Descrição |
|---|------|-----------|
| **F20** | Schwefel | Ótimos nas margens do domínio. Desestabiliza otimizadores que presumem convergência central Gaussiana. |
| **F21** | Gallagher 101 | 101 campânulas Gaussianas caóticas sobrepostas. Kriging O(N³) padece; Random Forest adapta nativamente. |
| **F22** | Gallagher 21 | 21 picos — versão mais branda de F21. Bacias mais largas facilitam regressão tangencial polinomial. |
| **F23** | Katsuura | Fractal severo baseado em produto. Correlação fractal modular destrói feature extraction estática. |
| **F24** | Lunacek bi-Rastrigin | **Deceptividade dupla**: falsa bacia ocupa 70% do espaço. Captura CMA-ES e Kriging ingênuo. |

In [ ]:
# ==============================================================================
# BBOB Base + 24 Functions
# ==============================================================================
class BBOBBase(Problem):
    def __init__(self, n_var=5, seed=0):
        super().__init__(n_var=n_var, n_obj=1,
                         xl=-5.0*np.ones(n_var), xu=5.0*np.ones(n_var))
        rng = np.random.RandomState(seed)
        self.x_opt = rng.uniform(-4, 4, n_var)
        self.R = ortho_group.rvs(n_var, random_state=rng)
        self.Q = ortho_group.rvs(n_var, random_state=rng)
        self.D = n_var
        self._cond = np.power(10.0, 6.0*np.arange(n_var)/max(n_var-1,1))

class BBOB_F1(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        out["F"] = np.sum((X-self.x_opt)**2, axis=1, keepdims=True)
class BBOB_F2(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        out["F"] = np.sum(self._cond*(X-self.x_opt)**2, axis=1, keepdims=True)
class BBOB_F3(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = X-self.x_opt; D=self.D
        out["F"] = (10*D+np.sum(Z**2-10*np.cos(2*np.pi*Z), axis=1, keepdims=True))
class BBOB_F4(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = X-self.x_opt; D=self.D; Zs = Z.copy()
        Zs[:,::2] = np.where(Zs[:,::2]>0, 10*Zs[:,::2], Zs[:,::2])
        c = np.power(10.0, 0.5*np.arange(D)/max(D-1,1)); Zc = c*Zs
        out["F"] = (10*D+np.sum(Zc**2-10*np.cos(2*np.pi*Zc), axis=1, keepdims=True))
class BBOB_F5(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        s = np.sign(self.x_opt)*np.power(10.0, np.arange(self.D)/max(self.D-1,1))
        out["F"] = np.sum(5*np.abs(s)-s*X, axis=1, keepdims=True)
class BBOB_F6(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T; Zm = Z.copy()
        Zm = np.where(Zm*self.x_opt[:self.D]>0, 100*Zm, Zm)
        c = np.power(10.0, 2.0*np.arange(self.D)/max(self.D-1,1))
        out["F"] = np.sum(c*Zm**2, axis=1, keepdims=True)
class BBOB_F7(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T; Zh = np.floor(0.5+Z); D=self.D
        c = np.power(10.0, 2.0*np.arange(D)/max(D-1,1))
        out["F"] = np.sum(c*Zh**2, axis=1, keepdims=True)
class BBOB_F8(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = np.maximum(1, np.sqrt(self.D)/8)*(X-self.x_opt)+1
        out["F"] = np.sum(100*(Z[:,:-1]**2-Z[:,1:])**2+(Z[:,:-1]-1)**2, axis=1, keepdims=True)
class BBOB_F9(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = np.maximum(1, np.sqrt(self.D)/8)*((X-self.x_opt)@self.R.T)+1
        out["F"] = np.sum(100*(Z[:,:-1]**2-Z[:,1:])**2+(Z[:,:-1]-1)**2, axis=1, keepdims=True)
class BBOB_F10(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T
        out["F"] = np.sum(self._cond*Z**2, axis=1, keepdims=True)
class BBOB_F11(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T
        out["F"] = (1e6*Z[:,0:1]**2+np.sum(Z[:,1:]**2, axis=1, keepdims=True))
class BBOB_F12(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T
        out["F"] = (Z[:,0:1]**2+1e6*np.sum(Z[:,1:]**2, axis=1, keepdims=True))
class BBOB_F13(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T
        out["F"] = (Z[:,0:1]**2+100*np.sqrt(np.sum(Z[:,1:]**2, axis=1, keepdims=True)+1e-30))
class BBOB_F14(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T; D=self.D
        e = 2+4*np.arange(D)/max(D-1,1)
        out["F"] = np.sqrt(np.sum(np.abs(Z)**e, axis=1, keepdims=True))
class BBOB_F15(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z = (X-self.x_opt)@self.R.T; D=self.D
        out["F"] = (10*D+np.sum(Z**2-10*np.cos(2*np.pi*Z), axis=1, keepdims=True))
class BBOB_F16(BBOBBase):
    def __init__(self, n_var=5, seed=16):
        super().__init__(n_var, seed)
        self._a,self._b,self._K=0.5,3,12
        self._f0=sum(self._a**k*np.cos(np.pi*self._b**k) for k in range(self._K))
    def _evaluate(self, X, out, *a, **k):
        Z=(X-self.x_opt)@self.R.T; D=self.D; v=np.zeros(X.shape[0])
        for i in range(D):
            for kk in range(self._K): v+=self._a**kk*np.cos(2*np.pi*self._b**kk*(Z[:,i]+0.5))
        out["F"] = (10*((v/D-self._f0)**3)).reshape(-1,1)
class BBOB_F17(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z=(X-self.x_opt)@self.R.T; s=np.sqrt(Z[:,:-1]**2+Z[:,1:]**2)
        out["F"] = (np.mean(np.sqrt(s)*(1+np.sin(50*np.power(s,0.2))**2), axis=1)**2).reshape(-1,1)
class BBOB_F18(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        c=np.power(10.0, 0.5*np.arange(self.D)/max(self.D-1,1))
        Z=c*((X-self.x_opt)@self.R.T); s=np.sqrt(Z[:,:-1]**2+Z[:,1:]**2)
        out["F"] = (np.mean(np.sqrt(s)*(1+np.sin(50*np.power(s,0.2))**2), axis=1)**2).reshape(-1,1)
class BBOB_F19(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z=np.maximum(1,np.sqrt(self.D)/8)*((X-self.x_opt)@self.R.T)+1
        s=100*(Z[:,:-1]**2-Z[:,1:])**2+(Z[:,:-1]-1)**2
        out["F"] = np.sum(s/4000-np.cos(s/np.sqrt(np.arange(1,s.shape[1]+1)))+1, axis=1, keepdims=True)
class BBOB_F20(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        Z=2*np.sign(self.x_opt)*X+self.x_opt; D=self.D
        Zh=np.clip(Z,-500,500)
        out["F"] = (418.9828872724339*D-np.sum(Zh*np.sin(np.sqrt(np.abs(Zh))), axis=1)).reshape(-1,1)
class BBOB_F21(BBOBBase):
    def __init__(self, n_var=5, seed=21):
        super().__init__(n_var, seed); rng=np.random.RandomState(seed+100)
        self._np=101; self._c=rng.uniform(-4.5,4.5,(101,n_var))
        self._w=np.concatenate([[10],rng.uniform(1,5,100)]); self._s=rng.uniform(0.3,1.5,101)
    def _evaluate(self, X, out, *a, **k):
        F=np.zeros(X.shape[0])
        for i in range(self._np): F-=self._w[i]*np.exp(-np.sum((X-self._c[i])**2, axis=1)/(2*self._s[i]**2))
        out["F"] = (F-F.min()+0.1).reshape(-1,1)
class BBOB_F22(BBOBBase):
    def __init__(self, n_var=5, seed=22):
        super().__init__(n_var, seed); rng=np.random.RandomState(seed+100)
        self._np=21; self._c=rng.uniform(-4.5,4.5,(21,n_var))
        self._w=np.concatenate([[10],rng.uniform(1,5,20)]); self._s=rng.uniform(0.5,2.0,21)
    def _evaluate(self, X, out, *a, **k):
        F=np.zeros(X.shape[0])
        for i in range(self._np): F-=self._w[i]*np.exp(-np.sum((X-self._c[i])**2, axis=1)/(2*self._s[i]**2))
        out["F"] = (F-F.min()+0.1).reshape(-1,1)
class BBOB_F23(BBOBBase):
    def _evaluate(self, X, out, *a, **k):
        D=self.D; Z=(X-self.x_opt)@self.R.T; res=np.ones(X.shape[0])
        for i in range(D):
            inner=np.zeros(X.shape[0])
            for j in range(1,33): inner+=np.abs(2**j*Z[:,i]-np.round(2**j*Z[:,i]))/2**j
            res*=(1+(i+1)*inner)**(10.0/D**1.2)
        out["F"] = ((10.0/D**2)*res-10.0/D**2).reshape(-1,1)
class BBOB_F24(BBOBBase):
    def __init__(self, n_var=5, seed=24):
        super().__init__(n_var, seed); D=n_var
        self._mu0=2.5; self._d=1.0
        self._s=1.0-1.0/(2.0*np.sqrt(D+20)-8.2)
        self._mu1=-np.sqrt((self._mu0**2-self._d)/self._s)
    def _evaluate(self, X, out, *a, **k):
        D=self.D; xh=np.where(X>=0,X,-X); Z=xh@self.R.T
        s1=np.sum((xh-self._mu0)**2, axis=1)
        s2=self._d*D+self._s*np.sum((xh-self._mu1)**2, axis=1)
        r=10*(D-np.sum(np.cos(2*np.pi*Z), axis=1))
        out["F"] = (np.minimum(s1,s2)+r).reshape(-1,1)

BBOB_CLASSES = {
    1:BBOB_F1,2:BBOB_F2,3:BBOB_F3,4:BBOB_F4,5:BBOB_F5,6:BBOB_F6,7:BBOB_F7,8:BBOB_F8,
    9:BBOB_F9,10:BBOB_F10,11:BBOB_F11,12:BBOB_F12,13:BBOB_F13,14:BBOB_F14,15:BBOB_F15,
    16:BBOB_F16,17:BBOB_F17,18:BBOB_F18,19:BBOB_F19,20:BBOB_F20,21:BBOB_F21,22:BBOB_F22,
    23:BBOB_F23,24:BBOB_F24,
}

In [ ]:
print('='*60); print('SUITE BBOB MONO (24 problemas, 5D)'); print('='*60)
bbob_mono = {}
for fid in range(1, 25):
    name = f'BBOB_F{fid}'
    prob = BBOB_CLASSES[fid](n_var=5, seed=fid)
    df = gen_sobol(prob, n_obs=50_000)
    dbest = find_best(df, 'f1', n=300)
    save_prob(name, df, dbest)
    bbob_mono[name] = (df, dbest)
    print(f"  {name}: {len(df):,} pts, best f1={dbest['f1'].min():.4f}")

### Distribuições de $f_1$ — Panorama das 24 Funções

O grid a seguir mostra a distribuição dos valores objetivos para cada uma das 24 funções BBOB.
A linha vermelha tracejada marca o limiar das top-300 melhores soluções encontradas.

Observe como funções multimodais (F3, F15, F20) apresentam distribuições com caudas pesadas,
enquanto funções unimodais suaves (F1, F2) concentram-se perto do ótimo.

A **linha preta vertical** em cada subplot marca $f^* = 0$, o ótimo global verdadeiro por construção.
Compare a distância entre a linha vermelha (melhor solução empírica encontrada) e a preta (ótimo teórico).

In [ ]:
plot_mono_grid(bbob_mono, n_cols=6, suptitle='BBOB Mono-Objetivo — Distribuição de $f_1$ (preto = $f^*=0$)')

In [ ]:
for fid in [1, 10, 16, 21, 24]:
    name = f'BBOB_F{fid}'
    df, dbest = bbob_mono[name]
    feats = [f'x_{i+1}' for i in range(5)]
    plot_dashboard(df, dbest, feats, f'{name} — Dashboard (5D)', n_obj=1)

---
# 6. Suíte BBOB-biobj — Bi-Objective COCO (Tusar *et al.*, 2016)

## Origem Histórica

Proposta por **Tea Tusar**, **Dimo Brockhoff**, **Nikolaus Hansen** e **Anne Auger** no artigo
*"COCO: The Bi-objective Black Box Optimization Benchmarking (bbob-biobj) Test Suite"* (2016),
apresentado no workshop COCO do GECCO.

## Princípio Combinatório

As 55 funções são geradas pela **combinação bi-objetivo** de **10 funções-base** da suíte BBOB
clássica mono-objetivo:

| ID | Função-Base | Classe |
|----|-------------|--------|
| F1 | Sphere | Separável |
| F2 | Ellipsoid Separável | Separável |
| F6 | Attractive Sector | Unimodal moderada |
| F8 | Rosenbrock | Unimodal moderada |
| F10 | Ellipsoid Rotated | Ill-conditioned |
| F11 | Discus | Ill-conditioned |
| F15 | Rastrigin Rotated | Multimodal c/ estrutura |
| F17 | Schaffer F7 | Multimodal c/ estrutura |
| F21 | Gallagher 101 Peaks | Multimodal fraca |
| F22 | Gallagher 21 Peaks | Multimodal fraca |

As 55 combinações correspondem a todos os pares únicos (incluindo auto-pares):
$\binom{10+1}{2} = 55$.

Cada par usa **instâncias diferentes** (seeds distintas) para que os ótimos de $f_1$ e $f_2$
estejam em **locais conflitantes** no espaço de decisão, criando trade-offs genuínos.

## Conflitos Inter-Objetivo Especialmente Desafiadores

| Combinação | Conflito |
|------------|----------|
| **F1/F1** (Sphere/Sphere) | Baseline: dois ótimos deslocados, trade-off linear simples |
| **F8/F8** (Rosenbrock/Rosenbrock) | Vales parabólicos perpendiculares — destrói partições ortogonais do XGBoost |
| **F10/F10** (Ellipsoid R/Ellipsoid R) | Condicionamento $10^6$ cruzado com rotações opostas |
| **F15/F17** (Rastrigin/Schaffer) | Multimodalidade rugosa vs. harmônicos de alta frequência — heterocedasticidade severa |
| **F21/F21** (Gallagher/Gallagher) | 202 bacias caóticas sobrepostas sem conectividade global |

In [ ]:
# ==============================================================================
# BBOB-biobj Factory
# ==============================================================================
class BBOBBiobj(Problem):
    def __init__(self, cls1, cls2, n_var=5, seed1=0, seed2=1):
        super().__init__(n_var=n_var, n_obj=2,
                         xl=-5.0*np.ones(n_var), xu=5.0*np.ones(n_var))
        self.p1 = cls1(n_var=n_var, seed=seed1)
        self.p2 = cls2(n_var=n_var, seed=seed2)
    def _evaluate(self, X, out, *args, **kwargs):
        f1 = eval_problem(self.p1, X).ravel()
        f2 = eval_problem(self.p2, X).ravel()
        out["F"] = np.column_stack([f1, f2])

BIOBJ_BASE_IDS = [1, 2, 6, 8, 10, 11, 15, 17, 21, 22]
BIOBJ_PAIRS = []
for i, a in enumerate(BIOBJ_BASE_IDS):
    for b in BIOBJ_BASE_IDS[i:]:
        BIOBJ_PAIRS.append((a, b))

print(f"Total de combinacoes BBOB-biobj: {len(BIOBJ_PAIRS)}")

In [ ]:
print('='*60); print('SUITE BBOB-BIOBJ (55 problemas, 5D)'); print('='*60)
print('Gerando fronteiras empiricas (50K) e de referencia (500K) para cada par...')
bbob_bi = {}
for idx, (a, b) in enumerate(BIOBJ_PAIRS):
    fname = f'biobj_F{idx+1}'
    label = f'F{a}/F{b}'
    prob = BBOBBiobj(BBOB_CLASSES[a], BBOB_CLASSES[b], n_var=5, seed1=a*10, seed2=b*10+1)
    df = gen_sobol(prob, n_obs=50_000)
    dfp = find_pareto(df, ['f1','f2'])
    tpf = get_true_pf_bbob_biobj(BBOB_CLASSES[a], BBOB_CLASSES[b],
                                  n_var=5, seed1=a*10, seed2=b*10+1, n_obs=500_000)
    save_prob(fname, df, dfp)
    bbob_bi[label] = (df, dfp, tpf)
    if (idx+1) % 10 == 0:
        print(f"  ... {idx+1}/55 concluidos")
print(f"  Concluido: {len(bbob_bi)} problemas BBOB-biobj")

### Grid das 55 Fronteiras de Pareto

Cada mini-gráfico mostra a fronteira empírica de um dos 55 pares. O rótulo indica as funções-base
combinadas (ex: F1/F2 = Sphere vs Ellipsoid Separável).

As fronteiras variam drasticamente: de trade-offs suaves quase lineares (F1/F1) a nuvens caóticas
sem estrutura clara (F21/F21). Esta variedade é proposital — testa se o SAEA generaliza
em conflitos inter-objetivo de todas as classes topológicas.

In [ ]:
plot_front_grid(bbob_bi, n_cols=8, suptitle='BBOB-biobj — 55 Fronteiras (preto = ref. 500K, vermelho = emp. 50K)',
                figw=3, figh=2.5)

In [ ]:
for label in ['F1/F1', 'F8/F8', 'F10/F10', 'F15/F17', 'F21/F21']:
    if label in bbob_bi:
        df, dfp, tpf = bbob_bi[label]
        feats = [f'x_{i+1}' for i in range(5)]
        plot_dashboard(df, dfp, feats, f'BBOB-biobj {label} — Dashboard (5D)', true_pf=tpf)

---
# Resumo Comparativo — 115 Problemas

A tabela a seguir consolida todos os problemas analisados neste notebook, incluindo:
- Dimensionalidade ($D$), número de objetivos ($M$)
- Tamanho da amostra gerada
- Tamanho do Front de Pareto empírico (ou Top-$n$ para mono-objetivo)

In [ ]:
rows = []
for name, entry in {**zdt, **dtlz, **wfg, **mmf}.items():
    df, dfp = entry[0], entry[1]
    tpf = entry[2] if len(entry) > 2 else None
    nobj = sum(1 for c in df.columns if c.startswith('f'))
    nvar = sum(1 for c in df.columns if c.startswith('x_'))
    pf_info = f'{len(tpf)} pts' if tpf is not None else 'N/A'
    rows.append((name, nvar, nobj, len(df), len(dfp), pf_info))
for name, entry in bbob_mono.items():
    rows.append((name, 5, 1, len(entry[0]), len(entry[1]), 'f*=0'))
for label, entry in bbob_bi.items():
    tpf = entry[2] if len(entry) > 2 else None
    pf_info = f'{len(tpf)} pts' if tpf is not None else 'N/A'
    rows.append((f'biobj {label}', 5, 2, len(entry[0]), len(entry[1]), pf_info))

df_sum = pd.DataFrame(rows, columns=['Problema','D','M','Amostras','Pareto Emp.','PF Verdadeira'])
print(f"Total de problemas analisados: {len(df_sum)}")
print(f"Total de amostras geradas: {df_sum['Amostras'].sum():,}")
display(df_sum.head(25))
print('...')
display(df_sum.tail(25))